# Notebook 16 — Near-Duplicate Leakage Screening



## Step 1 — Configuration

In [1]:
import os, re, time, json
import numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
import imagehash                       # pip install imagehash
import matplotlib.pyplot as plt

DATASET_ROOT = "/home/ali/Desktop/Algerian_Lamiaceae_v3"
OUT_DIR      = os.path.join(DATASET_ROOT, "results", "near_duplicates")
os.makedirs(OUT_DIR, exist_ok=True)
SPLITS   = ["train", "val", "test"]
IMG_EXT  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
AUG_RE   = re.compile(r"^aug_")       # offline-augmented copies (derived from train originals) are skipped
HASH_SIZE = 8                          # 8x8 -> 64-bit hashes
THRESHOLDS = [4, 6, 8, 10]             # Hamming distances reported (out of 64 bits)
MAIN_T     = 8                         # main threshold: pHash <= 8 AND dHash <= 12 counts as near-duplicate
DHASH_T    = 12
print(f"Dataset: {DATASET_ROOT}\nOutput : {OUT_DIR}")

Dataset: /home/ali/Desktop/Algerian_Lamiaceae_v3
Output : /home/ali/Desktop/Algerian_Lamiaceae_v3/results/near_duplicates


## Step 2 — Perceptual Hashes of All Original Images

pHash (frequency content) and dHash (gradients) are robust to resizing, mild cropping and recompression. Hashes are cached in `perceptual_hashes.csv`, so re-runs are fast.

In [2]:
cache = os.path.join(OUT_DIR, "perceptual_hashes.csv")
files = [(sp, c.name, f) for sp in SPLITS if (Path(DATASET_ROOT) / sp).exists()
         for c in sorted((Path(DATASET_ROOT) / sp).iterdir()) if c.is_dir()
         for f in sorted(c.iterdir()) if f.suffix.lower() in IMG_EXT and not AUG_RE.match(f.name)]
print(f"{len(files):,} original images to hash (augmented aug_* copies skipped)")

if os.path.exists(cache) and len(pd.read_csv(cache)) == len(files):
    H = pd.read_csv(cache, dtype={"phash": str, "dhash": str})
    print(f"Loaded cached hashes: {cache}")
else:
    rows, t0 = [], time.time()
    for i, (sp, cls, f) in enumerate(files, 1):
        try:
            im = Image.open(f).convert("RGB")
            rows.append({"split": sp, "cls": cls, "path": str(f),
                         "phash": str(imagehash.phash(im, hash_size=HASH_SIZE)),
                         "dhash": str(imagehash.dhash(im, hash_size=HASH_SIZE))})
        except Exception as e:
            print(f"  could not read {f}: {e}")
        if i % 1000 == 0 or i == len(files):
            print(f"  hashed {i:,}/{len(files):,} ({time.time() - t0:.0f}s)")
    H = pd.DataFrame(rows); H.to_csv(cache, index=False)
print(H.groupby(["split"]).size().to_string())

15,881 original images to hash (augmented aug_* copies skipped)
  hashed 1,000/15,881 (16s)
  hashed 2,000/15,881 (31s)
  hashed 3,000/15,881 (46s)
  hashed 4,000/15,881 (61s)
  hashed 5,000/15,881 (76s)
  hashed 6,000/15,881 (91s)
  hashed 7,000/15,881 (107s)
  hashed 8,000/15,881 (122s)
  hashed 9,000/15,881 (137s)
  hashed 10,000/15,881 (153s)
  hashed 11,000/15,881 (189s)
  hashed 12,000/15,881 (203s)
  hashed 13,000/15,881 (218s)
  hashed 14,000/15,881 (237s)
  hashed 15,000/15,881 (252s)
  hashed 15,881/15,881 (269s)
split
test      2366
train    11143
val       2372


## Step 3 — Find Near-Duplicate Pairs

Pairs with pHash distance ≤ 10 are collected and reported at several thresholds (4, 6, 8, 10 bits out of 64). The **main criterion** is pHash ≤ 8 **and** dHash ≤ 12, which keeps false positives low. Pairs inside the same split with **different labels** are also reported: they are the same image filed under two classes (label noise).

In [3]:
to_bits = lambda hexes: np.array([np.unpackbits(np.frombuffer(bytes.fromhex(h), np.uint8)) for h in hexes], bool)
P, D = to_bits(H.phash), to_bits(H.dhash)

def pairs_between(ia, ib, max_t, chunk=100):
    """All (i, j, pHash dist, dHash dist) with pHash distance <= max_t."""
    out = []
    for s in range(0, len(ia), chunk):
        a = ia[s:s + chunk]
        dp = (P[a][:, None, :] != P[ib][None, :, :]).sum(-1)
        ii, jj = np.nonzero(dp <= max_t)
        if len(ii):
            dd = (D[a[ii]] != D[ib[jj]]).sum(-1)
            out += list(zip(a[ii], ib[jj], dp[ii, jj], dd))
    return out

idx = {sp: np.where(H.split == sp)[0] for sp in SPLITS}
comparisons = [("test", "train"), ("val", "train"), ("test", "val")]
rows = []
for a, b in comparisons:
    t0 = time.time()
    found = pairs_between(idx[a], idx[b], max(THRESHOLDS))
    print(f"{a} vs {b}: {len(idx[a]):,} x {len(idx[b]):,} images -> {len(found)} candidate pairs ({time.time()-t0:.0f}s)")
    for i, j, dp, dd in found:
        rows.append({"pair": f"{a}-{b}", "img_a": H.path[i], "split_a": a, "cls_a": H.cls[i],
                     "img_b": H.path[j], "split_b": b, "cls_b": H.cls[j],
                     "phash_dist": int(dp), "dhash_dist": int(dd), "same_class": H.cls[i] == H.cls[j]})
# within-split pairs with DIFFERENT labels (same image filed under two classes = label noise)
for sp in SPLITS:
    for i, j, dp, dd in pairs_between(idx[sp], idx[sp], MAIN_T):
        if i < j and H.cls[i] != H.cls[j]:
            rows.append({"pair": f"{sp}-{sp}", "img_a": H.path[i], "split_a": sp, "cls_a": H.cls[i],
                         "img_b": H.path[j], "split_b": sp, "cls_b": H.cls[j],
                         "phash_dist": int(dp), "dhash_dist": int(dd), "same_class": False})

pairs = pd.DataFrame(rows, columns=["pair", "img_a", "split_a", "cls_a", "img_b", "split_b", "cls_b",
                                    "phash_dist", "dhash_dist", "same_class"])
pairs["near_duplicate"] = (pairs.phash_dist <= MAIN_T) & (pairs.dhash_dist <= DHASH_T)
pairs.sort_values(["phash_dist", "dhash_dist"]).to_csv(os.path.join(OUT_DIR, "near_duplicate_pairs.csv"), index=False)

summary = pd.DataFrame([{"comparison": f"{a} vs {b}", **{f"pHash<={t}": int((sub.phash_dist <= t).sum()) for t in THRESHOLDS},
                         "near_dup_pairs (main)": int(sub.near_duplicate.sum()),
                         "evaluation_images_affected": sub.loc[sub.near_duplicate, "img_a"].nunique()}
                        for a, b in comparisons for sub in [pairs[pairs.pair == f"{a}-{b}"]]])
print("\nNear-duplicate pairs by Hamming threshold:")
print(summary.to_string(index=False))
cross = pairs[pairs.near_duplicate & ~pairs.same_class]
print(f"\nNear-duplicates with DIFFERENT class labels (label noise): {len(cross)}")
if len(cross):
    print(cross[["split_a", "cls_a", "split_b", "cls_b", "phash_dist"]].to_string(index=False))
summary.to_csv(os.path.join(OUT_DIR, "near_duplicate_summary.csv"), index=False)

test vs train: 2,366 x 11,143 images -> 11 candidate pairs (3s)
val vs train: 2,372 x 11,143 images -> 6 candidate pairs (3s)
test vs val: 2,366 x 2,372 images -> 2 candidate pairs (1s)

Near-duplicate pairs by Hamming threshold:
   comparison  pHash<=4  pHash<=6  pHash<=8  pHash<=10  near_dup_pairs (main)  evaluation_images_affected
test vs train         1         2         2         11                      0                           0
 val vs train         0         0         0          6                      0                           0
  test vs val         0         0         0          2                      0                           0

Near-duplicates with DIFFERENT class labels (label noise): 0


## Step 4 — Visual Check

Look at the grid. If a flagged pair is clearly two *different* leaves, open `near_duplicate_pairs.csv`, set `near_duplicate` to `False` for that row, save, and re-run Step 5.

In [4]:
show = pairs[pairs.near_duplicate].sort_values("phash_dist").head(24)
if show.empty:
    print("No near-duplicate pairs at the main threshold.")
else:
    n = len(show); fig, axes = plt.subplots(n, 2, figsize=(5, 2.4 * n))
    axes = np.atleast_2d(axes)
    for r, (_, row) in enumerate(show.iterrows()):
        for k, (p, sp, c) in enumerate([(row.img_a, row.split_a, row.cls_a), (row.img_b, row.split_b, row.cls_b)]):
            ax = axes[r, k]; ax.imshow(Image.open(p).convert("RGB")); ax.axis("off")
            ax.set_title(f"{sp}/{c}\n{Path(p).name[:28]}", fontsize=6)
        axes[r, 0].text(-0.08, 0.5, f"pH={row.phash_dist}\ndH={row.dhash_dist}", transform=axes[r, 0].transAxes,
                        ha="right", va="center", fontsize=7)
    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "near_duplicate_pairs_grid.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Grid of the {n} closest pairs saved. Mark false positives in near_duplicate_pairs.csv "
          f"(set near_duplicate=False) and re-run Cell 5.")

No near-duplicate pairs at the main threshold.


## Step 5 — Leaked Evaluation Images

Test (and validation) images that have a near-duplicate in an earlier split. The list is saved to `leaked_test_images.csv`.

In [5]:
pairs = pd.read_csv(os.path.join(OUT_DIR, "near_duplicate_pairs.csv"))   # re-read: allows manual edits
leak = pairs[pairs.near_duplicate & pairs.pair.isin(["test-train", "val-train", "test-val"])]
leaked_test = sorted(set(leak.loc[leak.split_a == "test", "img_a"]))
leaked_val  = sorted(set(leak.loc[leak.split_a == "val", "img_a"]))
n_test = int((H.split == "test").sum())
pd.Series(leaked_test, name="path").to_csv(os.path.join(OUT_DIR, "leaked_test_images.csv"), index=False)
print(f"Test images with a near-duplicate in train/val: {len(leaked_test)} of {n_test} ({100*len(leaked_test)/max(n_test,1):.2f}%)")
print(f"Val images with a near-duplicate in train     : {len(leaked_val)}")
print(pd.Series([Path(p).parent.name for p in leaked_test]).value_counts().rename("leaked test images per class").to_string()
      if leaked_test else "  (none)")

Test images with a near-duplicate in train/val: 0 of 2366 (0.00%)
Val images with a near-duplicate in train     : 0
  (none)


## Step 6 — Re-Evaluate the CNN Without the Leaked Test Images

Uses the Notebook 14 checkpoint (`efficientnetb3_best.pth`) — no retraining. Metrics are macro-averaged, as in Notebook 14. The *Difference* row is the effect of removing the leaked images.

In [6]:
import torch, timm
import torchvision.datasets as datasets, torchvision.transforms as transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

CKPT = os.path.join(DATASET_ROOT, "results", "efficientnetb3_best.pth")
MODEL_NAME = "efficientnet_b3"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
test_ds = datasets.ImageFolder(os.path.join(DATASET_ROOT, "test"), transform=tf)
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=len(test_ds.classes))
model.load_state_dict(torch.load(CKPT, map_location=device)); model.to(device).eval()

probs, labels = [], []
with torch.no_grad():
    for k, (x, y) in enumerate(DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4), 1):
        probs.append(torch.softmax(model(x.to(device)), 1).cpu().numpy()); labels.append(y.numpy())
        if k % 20 == 0: print(f"  evaluated {k * 32:,}/{len(test_ds):,} test images")
probs, labels = np.concatenate(probs), np.concatenate(labels)
preds = probs.argmax(1)
leaked_set = {os.path.abspath(p) for p in leaked_test}
clean = np.array([os.path.abspath(p) not in leaked_set for p, _ in test_ds.samples])

def metrics(m):
    y, p, pr = labels[m], preds[m], probs[m]
    return {"n_images": int(m.sum()), "Accuracy": accuracy_score(y, p) * 100,
            "Macro_P": precision_score(y, p, average="macro", zero_division=0) * 100,
            "Macro_R": recall_score(y, p, average="macro", zero_division=0) * 100,
            "Macro_F1": f1_score(y, p, average="macro", zero_division=0) * 100,
            "Macro_AUC": roc_auc_score(y, pr, multi_class="ovr", average="macro",
                                       labels=list(range(len(test_ds.classes)))) * 100}
cnn_cmp = pd.DataFrame({"Full test set": metrics(np.ones(len(labels), bool)),
                        "Without near-duplicates": metrics(clean)}).T.round(2)
cnn_cmp.loc["Difference"] = (cnn_cmp.iloc[1] - cnn_cmp.iloc[0]).round(2)
print(cnn_cmp.to_string())
cnn_cmp.to_csv(os.path.join(OUT_DIR, "cnn_test_with_without_near_duplicates.csv"))

OutOfMemoryError: CUDA out of memory. Tried to allocate 222.00 MiB. GPU 0 has a total capacity of 7.65 GiB of which 37.69 MiB is free. Process 5497 has 1.15 GiB memory in use. Process 29697 has 5.90 GiB memory in use. Including non-PyTorch memory, this process has 552.00 MiB memory in use. Of the allocated memory 378.94 MiB is allocated by PyTorch, and 21.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Step 7 — Text for the Manuscript and Response Letter

Paste into p. 21 (lines 91–98) and adapt. If many test images are affected (more than ~1–2%), report the *without near-duplicates* numbers as the main CNN result.

In [ ]:
txt = (f"In addition to SHA-256 exact-duplicate removal, all {len(H):,} original images were screened for "
       f"near-duplicates (resized, cropped or recompressed copies) using 64-bit perceptual (pHash) and difference "
       f"(dHash) hashes; pairs with a pHash Hamming distance <= {MAIN_T} and dHash distance <= {DHASH_T} were flagged "
       f"and confirmed visually. {len(leaked_test)} of {n_test:,} test images "
       f"({100 * len(leaked_test) / max(n_test, 1):.2f}%) and {len(leaked_val)} validation images had a near-duplicate "
       f"in the training set, and {len(cross)} near-duplicate pair(s) carried different class labels. "
       f"Excluding the affected test images changed the CNN macro-F1 from {cnn_cmp.loc['Full test set', 'Macro_F1']:.2f}% "
       f"to {cnn_cmp.loc['Without near-duplicates', 'Macro_F1']:.2f}% (Table S_). Perceptual hashing does not detect "
       f"heavily edited or differently photographed copies of the same specimen, so residual leakage cannot be fully excluded.")
print(txt)
open(os.path.join(OUT_DIR, "near_duplicate_text.txt"), "w").write(txt)